# WAAM Twin Cloud Production Workflow

This notebook is intended for **Colab** and other cloud notebook / server environments where you want functionality close to local usage, but **headless**.

It supports:

- loading a real job YAML
- editing / overriding job parameters in the notebook
- running repeatable production-style simulations
- exporting VTK bundles and time series
- comparing parameter sweeps
- writing tuned job YAML files back out
- inspecting telemetry and scalar fields without the desktop GGUI viewer

This is the cloud analogue of your local workflow, not just a smoke test.

## 1. Repository and dependency setup

Use one of these approaches:

1. clone the **direct simulator repo** into the runtime,
2. upload / unzip the direct repo,
3. mount a persistent volume / network drive.

This notebook supports both layouts:

- direct repo root: `waam_solver/` cloned into a runtime folder such as `/content/waam_twin`
- nested local-dev layout: `FYP22-01/waam_twin/`

The cells below auto-detect both.

In [12]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/THEDARKDEACON/waam_solver.git"
CLONE_DIR = Path("/content/waam_twin")

if REPO_URL and not CLONE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)

candidates = [
    CLONE_DIR,
    Path("/content/waam_twin"),
    Path("/content/waam_solver"),
    Path("/content/FYP22-01/waam_twin"),
    Path("/content/FYP22-01"),
]

ROOT = next((p for p in candidates if p.exists() and (p / "requirements.txt").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate repo root. Expected a direct clone or nested FYP22-01/waam_twin layout.")

print("ROOT =", ROOT)
print("Exists:", ROOT.exists())
print("Repo URL =", REPO_URL)

ROOT = /content/waam_twin
Exists: True
Repo URL = https://github.com/THEDARKDEACON/waam_solver.git


In [13]:
# Install simulator dependencies from the detected repo root.

REQ = ROOT / "requirements.txt"
assert REQ.exists(), f"Missing requirements file: {REQ}"
%pip install -r {REQ}

In [14]:
import sys

# For the direct repo, the parent of ROOT must be on sys.path so the package
# name resolves from the cloned directory name.
PARENT = ROOT.parent
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

# Prefer CUDA when available. Set WAAM_BACKEND=cpu explicitly if you want a
# portable smoke run instead.
os.environ["WAAM_BACKEND"] = os.environ.get("WAAM_BACKEND", "cuda")
print("WAAM_BACKEND =", os.environ["WAAM_BACKEND"])
print("sys.path[0] =", sys.path[0])

WAAM_BACKEND = cuda
sys.path[0] = /content


In [15]:
# Optional: verify that CUDA is visible before Taichi initializes.
# Run this before the init cell if you want a quick environment check.

import subprocess

try:
    smi = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    )
    print("nvidia-smi:\n", smi)
except Exception as exc:
    print("nvidia-smi unavailable:", exc)

print("Requested WAAM_BACKEND =", os.environ.get("WAAM_BACKEND"))

nvidia-smi:
 Tesla T4, 15360, 14764, 0

Requested WAAM_BACKEND = cuda


## 2. Imports and runtime init

In [16]:
from waam_twin.platform import init_taichi
import os

profile = init_taichi(backend=os.environ["WAAM_BACKEND"])
profile

PlatformProfile(backend='cuda', device_name='cuda', vram_mb=15360, ram_mb=12975, tier='ultra')

In [17]:
import copy
import json
import math
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import yaml

from waam_twin.platform import init_taichi
try:
    from waam_twin.platform import reset_taichi
except ImportError:
    import taichi as ti

    def reset_taichi():
        ti.reset()
from waam_twin import WAAMTwin
from waam_twin.job import load_job_config

profile = init_taichi(backend=os.environ["WAAM_BACKEND"])
profile

PlatformProfile(backend='cuda', device_name='cuda', vram_mb=15360, ram_mb=12975, tier='ultra')

## 3. Load and inspect a baseline job

In [18]:
JOB_PATH = ROOT / "jobs" / "examples" / "bead_on_plate.yaml"
job = load_job_config(JOB_PATH)
job

{'simulation': {'preset': 'standard',
  'backend': 'cuda',
  'enable_vof': True,
  'enable_csf_tension': True,
  'enable_wetting': True,
  'enable_hydrostatic_gravity': True,
  'enable_bead_freeze': True,
  'enable_recoil': True,
  'use_recoil_clausius_clapeyron': True,
  'enable_lorentz': True,
  'enable_gas_shear': True,
  'enable_droplet_impact_pressure': True,
  'enable_enthalpy_cap': True,
  'arc_surface_weighting': True},
 'material': 'materials/validated/ER70S-6.v1.yaml',
 'calibration': 'materials/calibration/ER70S-6.bead_on_plate.yaml',
 'heat_source': 'goldak',
 'goldak': {'ff': 0.6, 'fr': 0.4, 'depth_front_mm': 0.9, 'depth_rear_mm': 1.8},
 'arc_physics': {'penetration_mm': 2.0,
  'T_vapor_cap_K': 3200.0,
  'surface_weighting': True},
 'advanced_physics': {'gas_jet_velocity_m_s': 12.0,
  'gas_shear_coeff': 1.0,
  'sigma_liquid_Sm': '1.0e6',
  'sigma_solid_Sm': '1.0e5',
  'lorentz_jacobi_iters': 40,
  'T_boiling_K': 3100.0,
  'L_vapor_J_kg': '6.0e6',
  'R_spec_vapor_J_kgK': 45

In [19]:
print(yaml.safe_dump(job, sort_keys=False))

simulation:
  preset: standard
  backend: cuda
  enable_vof: true
  enable_csf_tension: true
  enable_wetting: true
  enable_hydrostatic_gravity: true
  enable_bead_freeze: true
  enable_recoil: true
  use_recoil_clausius_clapeyron: true
  enable_lorentz: true
  enable_gas_shear: true
  enable_droplet_impact_pressure: true
  enable_enthalpy_cap: true
  arc_surface_weighting: true
material: materials/validated/ER70S-6.v1.yaml
calibration: materials/calibration/ER70S-6.bead_on_plate.yaml
heat_source: goldak
goldak:
  ff: 0.6
  fr: 0.4
  depth_front_mm: 0.9
  depth_rear_mm: 1.8
arc_physics:
  penetration_mm: 2.0
  T_vapor_cap_K: 3200.0
  surface_weighting: true
advanced_physics:
  gas_jet_velocity_m_s: 12.0
  gas_shear_coeff: 1.0
  sigma_liquid_Sm: 1.0e6
  sigma_solid_Sm: 1.0e5
  lorentz_jacobi_iters: 40
  T_boiling_K: 3100.0
  L_vapor_J_kg: 6.0e6
  R_spec_vapor_J_kgK: 450.0
process:
  current_A: 140
  voltage_V: 20
  arc_efficiency: 0.8
  travel_speed_mm_s: 15
  wire_feed_m_min: 8
  wire

## 4a. Run configuration and fidelity

These are the main notebook-side run controls:

- `PRESET_OVERRIDE`: overall fidelity preset, which controls domain size, target cell size, tracer budget, and solver mode
- `N_STEPS`: simulation duration
- `EXPORT_BUNDLE`, `EXPORT_SEQUENCE`, `SEQUENCE_EVERY`: output controls

If you have more compute available, start by moving from `standard` to `high` or `ultra`.

For deeper changes, edit `simulation.preset` in the job YAML or the preset definitions in `config/presets.yaml`.

In [20]:
PRESET_OVERRIDE = "high"   # None | "minimal" | "standard" | "high" | "ultra"
N_STEPS = 2000
EXPORT_BUNDLE = True
EXPORT_SEQUENCE = False
SEQUENCE_EVERY = 100

print({
    "PRESET_OVERRIDE": PRESET_OVERRIDE,
    "N_STEPS": N_STEPS,
    "EXPORT_BUNDLE": EXPORT_BUNDLE,
    "EXPORT_SEQUENCE": EXPORT_SEQUENCE,
    "SEQUENCE_EVERY": SEQUENCE_EVERY,
})

{'PRESET_OVERRIDE': 'high', 'N_STEPS': 2000, 'EXPORT_BUNDLE': True, 'EXPORT_SEQUENCE': False, 'SEQUENCE_EVERY': 100}


## 5a. Runtime preview frames

If you want an early visual diagnostic, use the helper below instead of a blind full run.

It advances the simulation step-by-step along the torch path and saves periodic mid-plane PNG previews so you can inspect bead shape, pool growth, and gross instability before committing to a long run.

In [21]:
def save_preview_slice_png(twin, out_path: Path, field: str = "Temperature_K", cmap: str = "inferno"):
    g = twin.grid
    if field == "Temperature_K":
        arr = g.T.to_numpy()
    elif field == "Liquid_Fraction":
        arr = g.f_l.to_numpy()
    elif field == "Speed_ms":
        ux = g.ux.to_numpy()
        uy = g.uy.to_numpy()
        uz = g.uz.to_numpy()
        arr = np.sqrt(ux * ux + uy * uy + uz * uz) * g.dx / g.dt
    else:
        raise KeyError(f"Unsupported preview field: {field}")

    mid_y = arr.shape[1] // 2
    sl = arr[:, mid_y, :].T
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 4))
    plt.imshow(sl, origin="lower", aspect="auto", cmap=cmap)
    plt.colorbar(label=field)
    plt.title(f"{field} at step {twin._step_n}")
    plt.xlabel("x cell")
    plt.ylabel("z cell")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    return out_path


def run_job_with_previews(
    job_dict: dict,
    *,
    preset_override: str | None = None,
    n_steps: int = 1200,
    preview_every: int = 200,
    max_preview_frames: int = 20,
    preview_field: str = "Temperature_K",
    preview_cmap: str = "inferno",
    out_dir: Path | None = None,
    tag: str | None = None,
):
    from waam_twin.job import load_job_config, parse_torch_path
    from waam_twin.torch_path import TorchPathDriver, clamp_torch_to_domain

    def _write_job_yaml_local(job_dict: dict, out_path: Path):
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(yaml.safe_dump(job_dict, sort_keys=False))
        return out_path

    run_root = (out_dir or ROOT / "cloud_runs")
    run_root.mkdir(parents=True, exist_ok=True)
    tag = tag or f"preview_{int(time.time())}"
    run_dir = run_root / tag
    run_dir.mkdir(parents=True, exist_ok=True)

    job_file = run_dir / f"{tag}.yaml"
    _write_job_yaml_local(job_dict, job_file)

    twin = WAAMTwin.from_job(job_file, preset_override=preset_override)
    twin.reset()
    job_cfg = load_job_config(job_file)
    g = twin.grid

    waypoints = parse_torch_path(job_cfg)
    if not waypoints:
        cy = (g.ny // 2) * g.dx
        waypoints = [(0.01, cy, 0.0)]
    driver = TorchPathDriver(waypoints, twin.travel_speed_m_s)

    interpass = int(getattr(twin, "_interpass_cooling_steps", 0) or 0)
    prev_seg = -1
    preview_dir = run_dir / "preview_frames"
    rows = []
    frame_idx = 0

    for step, x, y, z in driver.positions_for_steps(n_steps, g.dt):
        seg = driver.segment_index_at_distance(driver.distance_at_step(step, g.dt))
        if interpass > 0 and seg > prev_seg and prev_seg >= 0:
            park = driver.segment_end(prev_seg)
            px, py, pz = park if park else (x, y, z)
            for _ in range(interpass):
                cx, cy = clamp_torch_to_domain(px, py, g.nx, g.ny, g.dx)
                twin.step(cx, cy, is_welding=False, torch_z_m=pz)
        prev_seg = seg

        cx, cy = clamp_torch_to_domain(x, y, g.nx, g.ny, g.dx)
        twin.step(cx, cy, is_welding=True, torch_z_m=z)

        should_dump = (twin._step_n % preview_every == 0) or (step == n_steps - 1)
        if not should_dump:
            continue

        png_path = preview_dir / f"frame_{frame_idx:04d}_step_{twin._step_n:06d}.png"
        save_preview_slice_png(twin, png_path, field=preview_field, cmap=preview_cmap)
        telem = twin.get_telemetry()
        rows.append({
            "frame": frame_idx,
            "step": twin._step_n,
            "sim_time_ms": telem.get("sim_time_ms"),
            "pool_width_mm": telem.get("pool_width_mm"),
            "pool_depth_mm": telem.get("pool_depth_mm"),
            "bead_height_mm": telem.get("bead_height_mm"),
            "mass_balance_ratio": telem.get("mass_balance_ratio"),
            "png": str(png_path),
        })
        frame_idx += 1
        if frame_idx >= max_preview_frames:
            break

    preview_df = pd.DataFrame(rows)
    preview_csv = run_dir / "preview_manifest.csv"
    preview_df.to_csv(preview_csv, index=False)
    return {
        "run_dir": str(run_dir),
        "job_file": str(job_file),
        "preview_dir": str(preview_dir),
        "preview_manifest_csv": str(preview_csv),
        "preview_df": preview_df,
        "twin": twin,
    }

In [22]:
preview_result = run_job_with_previews(
    job,
    preset_override=PRESET_OVERRIDE,
    n_steps=min(N_STEPS, 1200),
    preview_every=150,
    max_preview_frames=8,
    preview_field="Temperature_K",
    tag="early_preview",
)
preview_result["preview_df"]

NameError: name 'write_job_yaml' is not defined

## 4. Utilities for tuning jobs in-notebook

Use `set_nested()` to override job YAML keys without manually editing dicts.

In [ ]:
def set_nested(d: dict, path: str, value):
    cur = d
    parts = path.split(".")
    for key in parts[:-1]:
        if key not in cur or not isinstance(cur[key], dict):
            cur[key] = {}
        cur = cur[key]
    cur[parts[-1]] = value


def make_job(base_job: dict, overrides: dict | None = None) -> dict:
    out = copy.deepcopy(base_job)
    for k, v in (overrides or {}).items():
        set_nested(out, k, v)
    return out


def write_job_yaml(job_dict: dict, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(job_dict, sort_keys=False))
    return out_path

## 5. Production-style single-run function

This function mirrors local usage:

- build the twin from a job,
- run a real path or step schedule,
- collect telemetry,
- optionally export bundle / sequence,
- return structured results for sweeps.

In [ ]:
def run_job(
    job_dict: dict,
    *,
    preset_override: str | None = None,
    n_steps: int = 1200,
    export_bundle: bool = False,
    export_sequence: bool = False,
    sequence_every: int = 100,
    out_dir: Path | None = None,
    tag: str | None = None,
):
    work_dir = ROOT / "jobs" / "generated"
    work_dir.mkdir(parents=True, exist_ok=True)
    run_root = (out_dir or ROOT / "cloud_runs")
    run_root.mkdir(parents=True, exist_ok=True)

    tag = tag or f"run_{int(time.time())}"
    run_dir = run_root / tag
    run_dir.mkdir(parents=True, exist_ok=True)

    job_file = run_dir / f"{tag}.yaml"
    write_job_yaml(job_dict, job_file)

    manifest_path = run_dir / "run_manifest.json"
    paths = {
        "run_dir": str(run_dir),
        "job_file": str(job_file),
        "manifest": str(manifest_path),
    }
    checks = {
        "requested_steps": int(n_steps),
        "bundle_ok": not export_bundle,
        "sequence_ok": not export_sequence,
    }
    warnings = []
    telem = {}
    twin = None
    elapsed = None

    def write_manifest(status: str, error: str | None = None, tb: str | None = None):
        manifest = {
            "tag": tag,
            "status": status,
            "completed": status.startswith("completed"),
            "elapsed_s": None if elapsed is None else round(elapsed, 3),
            "preset_override": preset_override,
            "n_steps": int(n_steps),
            "export_bundle": export_bundle,
            "export_sequence": export_sequence,
            "sequence_every": sequence_every,
            "job_file": str(job_file),
            "telemetry": telem,
            "paths": paths,
            "checks": checks,
            "warnings": warnings,
            "error": error,
            "traceback": tb,
        }
        manifest_path.write_text(json.dumps(manifest, indent=2))
        return manifest

    write_manifest("running")
    t0 = time.perf_counter()
    try:
        twin = WAAMTwin.from_job(job_file, preset_override=preset_override)
        twin.reset()
        twin.run_path(job_file, n_steps=n_steps)
        telem = twin.get_telemetry()
        elapsed = time.perf_counter() - t0

        if export_bundle:
            bundle_dir = run_dir / "bundle"
            paths.update(twin.export_research_bundle(bundle_dir, tag=tag))
            volume_path = paths.get("volume")
            checks["bundle_ok"] = bool(volume_path and Path(volume_path).exists())
            if not checks["bundle_ok"]:
                warnings.append("Bundle export requested but no volume VTI was found.")

        if export_sequence:
            seq_dir = run_dir / "sequence"
            expected_frames = max(1, n_steps // max(sequence_every, 1))
            vti_paths = twin.export_research_sequence(
                seq_dir,
                n_steps=n_steps,
                every_n=sequence_every,
                max_frames=expected_frames,
            )
            paths["sequence_dir"] = str(seq_dir)
            paths["sequence_vti_count"] = len(vti_paths)
            checks["expected_sequence_frames"] = expected_frames
            checks["sequence_ok"] = len(vti_paths) >= expected_frames
            if not checks["sequence_ok"]:
                warnings.append(
                    f"Sequence export requested {expected_frames} frames but wrote {len(vti_paths)}."
                )

        mass_ratio = telem.get("mass_balance_ratio")
        checks["mass_balance_ratio_finite"] = bool(np.isfinite(mass_ratio)) if mass_ratio is not None else False
        if not checks["mass_balance_ratio_finite"]:
            warnings.append("mass_balance_ratio is missing or non-finite.")

        status = "completed_with_warnings" if warnings else "completed"
        manifest = write_manifest(status)
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        manifest = write_manifest("failed", error=repr(exc), tb=traceback.format_exc())
        raise RuntimeError(f"Run '{tag}' failed. See {manifest_path}") from exc

    return {
        "status": manifest["status"],
        "job_file": str(job_file),
        "elapsed_s": round(elapsed, 3),
        "telemetry": telem,
        "paths": paths,
        "checks": checks,
        "warnings": warnings,
        "manifest": manifest,
        "twin": twin,
    }

## 6. Optional interactive controls

If `ipywidgets` is available, you can tune a few common process parameters without editing Python dicts manually.

This does not replace YAML-based jobs; it is just a convenience layer on top of them.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception as exc:
    HAS_WIDGETS = False
    print("ipywidgets unavailable:", exc)

if HAS_WIDGETS:
    w_travel = widgets.FloatSlider(value=float(job["process"].get("travel_speed_mm_s", 15.0)), min=2.0, max=20.0, step=0.5, description="travel")
    w_wfs = widgets.FloatSlider(value=float(job["process"].get("wire_feed_m_min", 8.0)), min=2.0, max=15.0, step=0.25, description="wfs")
    w_current = widgets.FloatSlider(value=float(job["process"].get("current_A", 140.0)), min=80.0, max=260.0, step=5.0, description="current")
    w_mode = widgets.Dropdown(options=["auto", "globular", "spray", "pulsed"], value=str(job["process"].get("transfer_mode", "auto")), description="mode")
    w_angle = widgets.FloatSlider(value=float(job["process"].get("impact_lead_angle_deg", 8.0)), min=0.0, max=25.0, step=1.0, description="lead")
    controls = widgets.VBox([w_travel, w_wfs, w_current, w_mode, w_angle])
    display(controls)

    def job_from_widgets(base_job: dict) -> dict:
        return make_job(base_job, {
            "process.travel_speed_mm_s": float(w_travel.value),
            "process.wire_feed_m_min": float(w_wfs.value),
            "process.current_A": float(w_current.value),
            "process.transfer_mode": str(w_mode.value),
            "process.impact_lead_angle_deg": float(w_angle.value),
        })
else:
    def job_from_widgets(base_job: dict) -> dict:
        return copy.deepcopy(base_job)

## 7. Run the current widget / baseline configuration

In [ ]:
job_widget = job_from_widgets(job)
widget_result = run_job(
    job_widget,
    preset_override=PRESET_OVERRIDE,
    n_steps=N_STEPS,
    export_bundle=EXPORT_BUNDLE,
    export_sequence=EXPORT_SEQUENCE,
    sequence_every=SEQUENCE_EVERY,
    tag="widget_run",
)
widget_result["telemetry"]

## 6. Example: tune one job in the notebook

Edit the `overrides` dict below to tune the same kinds of parameters you change locally.

In [ ]:
overrides = {
    "calibration": None,
    "process.travel_speed_mm_s": 8.0,
    "process.transfer_mode": "spray",
    "process.impact_lead_angle_deg": 6.0,
    "deposition.trailing_solidify_lookback_mm": 2.0,
    "deposition.trailing_solidify_temp_margin_K": 30.0,
}

job_tuned = make_job(job, overrides)
print(yaml.safe_dump(job_tuned, sort_keys=False))

In [ ]:
result = run_job(
    job_tuned,
    preset_override=PRESET_OVERRIDE,
    n_steps=N_STEPS,
    export_bundle=EXPORT_BUNDLE,
    export_sequence=EXPORT_SEQUENCE,
    sequence_every=SEQUENCE_EVERY,
    tag="tuned_single",
)
result["telemetry"]

## 7. Batch sweeps for tuning

This is the part that makes cloud notebooks genuinely useful for production-style job tuning.

In [ ]:
def flatten_result(label: str, result: dict, overrides: dict) -> dict:
    row = {
        "label": label,
        "elapsed_s": result["elapsed_s"],
        **overrides,
    }
    row.update(result["telemetry"])
    return row


cases = [
    ("globular_5mms", {
        "calibration": None,
        "process.travel_speed_mm_s": 5.0,
        "process.transfer_mode": "globular",
    }),
    ("spray_8mms", {
        "calibration": None,
        "process.travel_speed_mm_s": 8.0,
        "process.transfer_mode": "spray",
    }),
    ("spray_12mms", {
        "calibration": None,
        "process.travel_speed_mm_s": 12.0,
        "process.transfer_mode": "spray",
    }),
]

rows = []
for label, ov in cases:
    job_case = make_job(job, ov)
    res = run_job(job_case, n_steps=1000, tag=label)
    rows.append(flatten_result(label, res, ov))

df = pd.DataFrame(rows)
df[[
    "label",
    "process.travel_speed_mm_s",
    "process.transfer_mode",
    "pool_depth_mm",
    "bead_height_mm",
    "mass_balance_ratio",
    "droplet_transfer_mode",
    "droplet_impact_velocity_ms",
    "elapsed_s",
]]

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["label"], df["bead_height_mm"], marker="o", label="bead_height_mm")
plt.plot(df["label"], df["pool_depth_mm"], marker="s", label="pool_depth_mm")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(df["label"], df["mass_balance_ratio"], marker="o")
plt.axhline(1.0, color="red", linestyle="--", label="ideal")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Rank sweep results

This gives you a simple production-style score so you can sort candidate jobs instead of manually eyeballing every run.

In [ ]:
# Adjust these targets for your study.
TARGETS = {
    "bead_height_mm": 1.2,
    "pool_depth_mm": 2.8,
    "mass_balance_ratio": 1.0,
}
WEIGHTS = {
    "bead_height_mm": 1.0,
    "pool_depth_mm": 1.0,
    "mass_balance_ratio": 1.5,
}

rank_df = df.copy()
score = np.zeros(len(rank_df), dtype=float)
for key, target in TARGETS.items():
    if key not in rank_df:
        continue
    err = np.abs(rank_df[key] - target) / max(abs(target), 1e-9)
    score += WEIGHTS.get(key, 1.0) * err
rank_df["score"] = score
rank_df = rank_df.sort_values("score").reset_index(drop=True)
rank_df[[
    "label",
    "score",
    "bead_height_mm",
    "pool_depth_mm",
    "mass_balance_ratio",
    "droplet_transfer_mode",
    "droplet_impact_velocity_ms",
    "elapsed_s",
]]

## 11. List saved cloud runs

Every `run_job(...)` call writes a `run_manifest.json` into a dedicated run directory. This lets you inspect prior server runs without re-running them.

In [ ]:
run_root = ROOT / "cloud_runs"
manifest_paths = sorted(run_root.glob("*/run_manifest.json"))
rows = []
for p in manifest_paths:
    data = json.loads(p.read_text())
    row = {
        "tag": data.get("tag"),
        "status": data.get("status"),
        "elapsed_s": data.get("elapsed_s"),
        "n_steps": data.get("n_steps"),
        "run_dir": str(Path(data["paths"]["run_dir"])),
    }
    row.update(data.get("telemetry", {}))
    rows.append(row)

history_df = pd.DataFrame(rows)
history_df.sort_values(["status", "tag"]) if not history_df.empty else history_df

## 8. Export sequence runs for ParaView / server post-processing

This is the closest cloud equivalent to a local inspection workflow.

In [ ]:
sequence_result = run_job(
    job_tuned,
    preset_override=PRESET_OVERRIDE,
    n_steps=N_STEPS,
    export_bundle=False,
    export_sequence=True,
    sequence_every=SEQUENCE_EVERY,
    tag="tuned_sequence",
)
sequence_result["paths"]

## 9. Inspect bundle / VTK arrays in-notebook

In [ ]:
bundle = result["paths"]
vti_path = Path(bundle["volume"])
grid = pv.read(str(vti_path))

print("VTI:", vti_path)
print("n_cells:", grid.n_cells)
print("spacing:", grid.spacing)
print("arrays:", sorted(grid.cell_data.keys()))

for name in ["Temperature_K", "Liquid_Fraction", "Speed_ms", "Curvature_kappa", "BodyForce_Z_ms2"]:
    if name in grid.cell_data:
        arr = np.asarray(grid.cell_data[name])
        print(name, "min=", np.nanmin(arr), "max=", np.nanmax(arr), "mean=", np.nanmean(arr))

In [ ]:
def show_mid_y_slice(grid, field="Temperature_K", cmap="inferno"):
    dims = tuple(int(v - 1) for v in grid.dimensions)
    arr = np.asarray(grid.cell_data[field]).reshape(dims, order="F")
    mid_y = arr.shape[1] // 2
    sl = arr[:, mid_y, :].T
    plt.figure(figsize=(11, 4))
    plt.imshow(sl, origin="lower", aspect="auto", cmap=cmap)
    plt.colorbar(label=field)
    plt.title(f"Mid-Y slice: {field}")
    plt.xlabel("x cell")
    plt.ylabel("z cell")
    plt.tight_layout()
    plt.show()


show_mid_y_slice(grid, "Temperature_K")
show_mid_y_slice(grid, "Liquid_Fraction", cmap="viridis")

## 10. Save tuned jobs back to disk

This is important for production use: the notebook should produce job YAMLs you can rerun locally or on servers.

In [ ]:
tuned_jobs_dir = ROOT / "jobs" / "tuned"
saved_job = write_job_yaml(job_tuned, tuned_jobs_dir / "bead_on_plate.cloud_tuned.yaml")
print(saved_job)
print(saved_job.read_text())

## 11. Recommended cloud/server workflow

For production-grade server use, I would structure it like this:

1. **Notebook for development / tuning**
   - edit YAML-derived parameters
   - run batches
   - inspect telemetry / VTK quickly

2. **Saved tuned job YAMLs**
   - `jobs/tuned/*.yaml`
   - source of truth for server runs

3. **Headless batch scripts**
   - `python -m waam_twin.export ...`
   - or Python loops calling `WAAMTwin.from_job(...)`

4. **Post-processing**
   - VTK bundles / sequences
   - ParaView or Python-based slicing

This gives you essentially the same core functionality as local use, except the desktop GGUI viewer is replaced by notebook plots and VTK inspection.

## 12. Optional next upgrade

If you want this to become fully production-like, the next notebook upgrade should be:

- interactive parameter widgets (`ipywidgets`)
- sweep result tables with sorting / filtering
- automatic export naming and run manifests
- comparison against `reference` / `model_reference`
- one-click generation of `jobs/tuned/*.yaml`
- job queue integration for remote compute nodes